# Channel A: Event Hubs → Bronze 스트리밍 확인

- 일반 컴퓨트에서 기존 소스·테이블·체크포인트를 유지하고 processingTime으로 실행한다.
- 송신기가 출력한 test_run_id로 이번 실행의 6개 이벤트를 확인한다.
- 로컬에서는 구조·Python 문법만 검증했다. Databricks 실행 결과는 아직 없다.

## 실행 전 확인

- 기존 노트북/Job의 동일 체크포인트 쿼리를 먼저 중지한다. 체크포인트는 삭제하지 않는다.
- 기존 Kafka 서비스 자격 증명이 이 컴퓨트에서도 동작하는지 확인한다. databricks.serviceCredential은 DBR 16.1 이상이 필요하다.
- Event Hubs 읽기, 서비스 자격 증명 사용, Volume 읽기/쓰기, Bronze 테이블 생성/추가 권한이 필요하다.
- 아래 호스트·Hub·서비스 자격 증명은 제공된 노트북의 값이다. 현재 리소스와 다르면 수정한다.


In [ ]:
from pyspark.sql import functions as F

TABLE_NAME = "commerce_poc.bronze.events"
CHECKPOINT_PATH = "/Volumes/commerce_poc/bronze/checkpoints/event_hubs_to_bronze"
QUERY_NAME = "event_hubs_to_bronze_live"

kafka_options = {
    "kafka.bootstrap.servers": "EVH-training-demo-krc-swlee.servicebus.windows.net:9093",
    "subscribe": "commerce-events",
    "databricks.serviceCredential": "commerce_event_hubs_reader",
    "startingOffsets": "earliest",
}

spark.conf.set("spark.sql.session.timeZone", "UTC")
dbutils.widgets.text("test_run_id", "", "송신기의 test_run_id")
spark.sql("CREATE VOLUME IF NOT EXISTS commerce_poc.bronze.checkpoints")


## 원문 적재 시작

- raw_payload와 기존 출처 컬럼을 그대로 저장한다. 이 단계에서는 Silver 변환을 하지 않는다.
- 5초는 새 데이터를 확인하는 간격이다. 적재 완료 시간이 5초 이내라는 보장은 아니다.

startingOffsets는 새 체크포인트로 시작할 때 적용된다. 기존 체크포인트가 있으면 저장된 오프셋에서 이어진다.
아래 셀은 awaitTermination으로 막지 않으므로 이후 셀에서 상태를 확인할 수 있다.


In [ ]:
if any(query.name == QUERY_NAME for query in spark.streams.active):
    raise RuntimeError("이 세션에 같은 이름의 쿼리가 이미 실행 중입니다. 먼저 상태를 확인하세요.")

events_df = spark.readStream.format("kafka").options(**kafka_options).load()
bronze_df = (
    events_df.selectExpr(
        "cast(value as string) as raw_payload",
        "topic as source_event_hub",
        "partition",
        "offset",
        "timestamp as broker_timestamp",
    )
    .withColumn("bronze_ingested_at", F.current_timestamp())
)

bronze_query = (
    bronze_df.writeStream.format("delta")
    .queryName(QUERY_NAME)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="5 seconds")
    .toTable(TABLE_NAME)
)
print("active:", bronze_query.isActive)


## 송신 중 상태 확인

- 이 셀은 필요할 때 다시 실행한다. isActive=True와 실제 입력 처리를 함께 확인한다.
- idle 배치의 numInputRows=0은 정상일 수 있다. 에러는 exception과 로그로 확인한다.

이제 로컬 송신기를 실행한다. 송신기에서 출력된 test_run_id를 상단 위젯에 넣고 결과 셀을 반복 실행한다.


In [ ]:
print("active:", bronze_query.isActive)
print("status:", bronze_query.status)
print("exception:", bronze_query.exception())
for progress in bronze_query.recentProgress[-5:]:
    print({key: progress.get(key) for key in ("batchId", "timestamp", "numInputRows")})


## 이번 실행의 Bronze 결과 확인

- 전체 테스트는 6개 event_id, 단건 테스트는 a-poc-v1-01 한 개를 기대한다.
- 부분 도착 상태에서 완료로 판단하지 않는다. 중복 원문이 있으면 별도로 보고한다.

published_at은 HTTP 송신 직전 시각이고 occurred_at은 고정 fixture의 업무 시각이다.
bronze_ingested_at은 micro-batch 평가 시각이며 Delta 커밋 완료 시각이 아니다. 이번 확인으로 지연시간 SLA를 주장하지 않는다.


In [ ]:
run_id = dbutils.widgets.get("test_run_id").strip()
EXPECTED_COUNT = 6  # --limit 1 단건 테스트를 확인할 때만 1로 바꾼다.

if not run_id:
    print("상단 test_run_id 위젯에 송신기가 출력한 값을 입력하세요.")
else:
    run_rows = (
        spark.table(TABLE_NAME)
        .filter(F.get_json_object("raw_payload", "$.test_run_id") == F.lit(run_id))
        .select(
            F.get_json_object("raw_payload", "$.event_id").alias("event_id"),
            F.get_json_object("raw_payload", "$.event_name").alias("event_name"),
            F.get_json_object("raw_payload", "$.order_no").alias("order_no"),
            F.get_json_object("raw_payload", "$.published_at").alias("published_at"),
            "source_event_hub", "partition", "offset", "broker_timestamp", "bronze_ingested_at", "raw_payload",
        )
        .cache()
    )
    try:
        count = run_rows.count()
        observed = {row.event_id for row in run_rows.select("event_id").distinct().collect()}
        expected = {f"a-poc-v1-{index:02d}" for index in range(1, EXPECTED_COUNT + 1)}
        print({"row_count": count, "distinct_event_ids": len(observed),
               "missing_ids": sorted(expected - observed), "unexpected_ids": sorted(observed - expected),
               "duplicate_rows": count - len(observed),
               "six_or_one_row_check_passed": observed == expected and count == EXPECTED_COUNT})
        display(run_rows.orderBy("event_id").limit(20))
    finally:
        run_rows.unpersist()


## 확인 후 중지

- 모든 결과 확인 후 아래 STOP_STREAM을 True로 바꿔 실행한다.
- 테스트 중에는 False를 유지한다. 체크포인트를 보존해야 다음 실행에서 이어서 읽을 수 있다.

동일 파티션의 offset은 메시지 위치다. 여러 파티션의 offset을 전역 이벤트 순서로 해석하지 않는다.


In [ ]:
STOP_STREAM = False
if STOP_STREAM:
    bronze_query.stop()
    print("active:", bronze_query.isActive)
